## Lab6-Assignment: Topic Classification

Use the same training, development, and test partitions of the the 20 newsgroups text dataset as in Lab6.4-Topic-classification-BERT.ipynb 

* Fine-tune and examine the performance of another transformer-based pretrained language models, e.g., RoBERTa, XLNet

* Compare the performance of this model to the results achieved in Lab6.4-Topic-classification-BERT.ipynb and to a conventional machine learning approach (e.g., SVM, Naive Bayes) using bag-of-words or other engineered features of your choice. 
Describe the differences in performance in terms of Precision, Recall, and F1-score evaluation metrics.

## Assignment implementation: fine-tune RoBERTa for topic classification

In this section we fine-tune `roberta-base` on the same 4-topic subset of the 20 newsgroups dataset used in `Lab6.4-Topic-classification-BERT.ipynb`.

The notebook below installs the required packages, prepares train/dev/test splits, fine-tunes the model, and reports precision, recall, and F1-score on the test set.

In [7]:
!pip install -q pandas numpy scikit-learn simpletransformers torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\smart\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs

# Load the same 4 categories used in the BERT notebook
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'sci.space']

newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)

train_df = pd.DataFrame({
    'text': newsgroups_train.data,
    'labels': newsgroups_train.target
})
test_df = pd.DataFrame({
    'text': newsgroups_test.data,
    'labels': newsgroups_test.target
})

train_df, dev_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df[['labels']]
)

print('Train size:', len(train_df))
print('Dev size:', len(dev_df))
print('Test size:', len(test_df))
print('Labels:', train_df['labels'].value_counts().sort_index().to_dict())

ModuleNotFoundError: No module named 'pandas'

In [9]:
model_args = ClassificationArgs()
model_args.overwrite_output_dir = True
model_args.evaluate_during_training = True
model_args.evaluate_during_training_steps = 32
model_args.save_eval_checkpoints = False
model_args.save_model_every_epoch = False
model_args.num_train_epochs = 3
model_args.train_batch_size = 16
model_args.eval_batch_size = 32
model_args.learning_rate = 4e-5
model_args.max_seq_length = 256
model_args.use_multiprocessing = False
model_args.use_multiprocessing_for_evaluation = False
model_args.no_cache = True

model = ClassificationModel(
    'roberta',
    'roberta-base',
    num_labels=4,
    args=model_args,
    use_cuda=False
)

model.train_model(train_df, eval_df=dev_df)

NameError: name 'ClassificationArgs' is not defined

In [ ]:
# Evaluate on the development set
result_dev, _, _ = model.eval_model(dev_df)
print('Dev evaluation results:')
print(result_dev)

# Predict on the test set
predictions, _ = model.predict(test_df['text'].tolist())
test_df['predicted'] = predictions

print('Test set classification report:')
print(classification_report(test_df['labels'], test_df['predicted'], target_names=newsgroups_train.target_names))

### Notes

* This notebook fine-tunes `roberta-base` instead of `bert-base-cased`.
* The key performance metrics are Precision, Recall and F1-score on the held-out test set.
* To complete the full assignment, compare these results to the BERT results in `Lab6.4-Topic-classification-BERT.ipynb` and to a conventional baseline such as SVM or Naive Bayes with bag-of-words features.